[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/06_mha_solution.ipynb)

# 🔴 Solution: Multi-Head Attention

*Attention & Transformers · Hard*

Reference implementation. Try it yourself in `06_mha.ipynb` first.

---
Implement **multi-head attention**.

$$\text{head}_i = \text{softmax}\!\left(\frac{Q W^Q_i (K W^K_i)^\top}{\sqrt{d_k}}\right)V W^V_i$$

$$\text{MHA}(Q,K,V) = \text{Concat}(\text{head}_1..\text{head}_H)\,W^O$$

### Signature
```python
class MultiHeadAttention(nnx.Module):
    def __init__(self, d_model: int, num_heads: int, *, rngs: nnx.Rngs): ...
    def __call__(self, Q, K, V): ...
```

### Requirements
- Use `nnx.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `self.d_k = d_model // num_heads`
- `Q` is `(B, seq_q, d_model)`; `K` and `V` are `(B, seq_k, d_model)`
- Must support **cross-attention** — `seq_q != seq_k`
- Do **not** use `nnx.MultiHeadAttention`

`nnx.Linear` is an allowed building block: you are implementing attention, not
the projection. It also brings its own initialization.

### Heads are a reshape, not a loop
The whole trick is that $H$ separate attention computations are one batched
computation. `(B, seq, d_model)` reshapes to `(B, seq, H, d_k)` and transposes to
`(B, H, seq, d_k)`, after which the head axis is just another batch axis and the
same einsum handles all of them. Nothing is looped, and the parameter count is
identical to single-head attention with the same `d_model` — you are
partitioning the projection, not adding to it.

### Why Q, K and V are separate arguments
Passing one `x` would only ever give you self-attention. Taking three inputs
means the identical class does self-attention (`mha(x, x, x)`) and
cross-attention (`mha(decoder, encoder, encoder)`), which is exactly how an
encoder-decoder transformer reuses one implementation.

### The trap
It is tempting to read a single `seq` off `Q` and use it for `K` too. That works
for every self-attention test and then fails the moment the sequence lengths
differ — so the score matrix is `(seq_q, seq_k)`, not square.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from flax import nnx


class MultiHeadAttention(nnx.Module):
    def __init__(self, d_model: int, num_heads: int, *, rngs: nnx.Rngs):
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nnx.Linear(d_model, d_model, rngs=rngs)
        self.W_k = nnx.Linear(d_model, d_model, rngs=rngs)
        self.W_v = nnx.Linear(d_model, d_model, rngs=rngs)
        self.W_o = nnx.Linear(d_model, d_model, rngs=rngs)

    def _split(self, t, B, seq):
        # (B, seq, d_model) -> (B, H, seq, d_k): heads become a batch axis.
        return t.reshape(B, seq, self.num_heads, self.d_k).transpose(0, 2, 1, 3)

    def __call__(self, Q, K, V):
        B, seq_q, _ = Q.shape
        seq_k = K.shape[1]        # NOT seq_q — this is what allows cross-attention

        q = self._split(self.W_q(Q), B, seq_q)
        k = self._split(self.W_k(K), B, seq_k)
        v = self._split(self.W_v(V), B, seq_k)

        # == q @ jnp.swapaxes(k, -1, -2)
        scores = jnp.einsum("bhqd,bhkd->bhqk", q, k) / jnp.sqrt(
            jnp.asarray(self.d_k, Q.dtype)
        )
        weights = jax.nn.softmax(scores, axis=-1)
        attn = jnp.einsum("bhqk,bhkd->bhqd", weights, v)      # == weights @ v

        out = attn.transpose(0, 2, 1, 3).reshape(B, seq_q, -1)
        return self.W_o(out)

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp
from flax import nnx

mha = MultiHeadAttention(32, 4, rngs=nnx.Rngs(params=0))

x = jax.random.normal(jax.random.key(1), (2, 6, 32))
print("self-attention :", mha(x, x, x).shape)

ctx = jax.random.normal(jax.random.key(2), (2, 10, 32))
print("cross-attention:", mha(x, ctx, ctx).shape, "(query length wins)")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("mha")